In [1]:
url='http://www.lps.usp.br/hae/apostila/feiCorCrop.zip'
import os; nomeArq=os.path.split(url)[1]
if not os.path.exists(nomeArq):
  print("Baixando o arquivo",nomeArq,"para diretorio default",os.getcwd())
  os.system("wget -nc -U 'Firefox/50.0' "+url)
else:
  print("O arquivo",nomeArq,"ja existe no diretorio default",os.getcwd())
print("Descompactando arquivos novos de",nomeArq)
os.system("unzip -u "+nomeArq)

O arquivo feiCorCrop.zip ja existe no diretorio default /content
Descompactando arquivos novos de feiCorCrop.zip


0

In [3]:
# fei_resnet_5xvalidation_melhorado.py
# Classificação M/F com ResNet50 e 5-fold cross validation
#
# Melhorias feitas para aumentar a acurácia:
# - camada superior melhor:
#     GlobalAveragePooling2D
#     BatchNormalization
#     Dense(256, relu)
#     Dropout(0.5)
# - separação de parte do treino para validação
# - uso de callbacks:
#     ModelCheckpoint
#     EarlyStopping
#     ReduceLROnPlateau
# - data augmentation leve
# - fine tuning conservador:
#     libera só as últimas 10 camadas do modelo-base
#     recompila com Adam(1e-6)
#
# Observação:
# O programa executa apenas 1 repetição de 5-fold cross validation:
#     for repete_no in range(1)
# como sugerido no enunciado para não gastar muito tempo.

import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
os.environ['TF_FORCE_GPU_ALLOW_GROWTH'] = 'true'

import numpy as np
import tensorflow.keras as keras

from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D, BatchNormalization
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.applications.resnet50 import ResNet50, preprocess_input
from tensorflow.keras.preprocessing import image
from tensorflow.keras.preprocessing.image import ImageDataGenerator

from sklearn.model_selection import StratifiedKFold, train_test_split


def leCsv(nomeDir, nomeArq, nl=0, nc=0):
    """
    Lê um arquivo CSV no formato:
        nome_da_imagem;rotulo

    rotulo:
        0 = masculino
        1 = feminino

    As imagens são carregadas, redimensionadas para nl x nc
    e pré-processadas com preprocess_input da ResNet50.
    """
    st = os.path.join(nomeDir, nomeArq)

    with open(st, "rt") as arq:
        lines = arq.readlines()

    n = len(lines)

    linhas_separadas = []
    for linha in lines:
        linha = linha.strip('\n')
        linha = linha.split(';')
        linhas_separadas.append(linha)

    y = np.empty((n,), dtype='float32')
    x = np.empty((n, nl, nc, 3), dtype='float32')

    for i in range(n):
        linha = linhas_separadas[i]
        img_path = os.path.join(nomeDir, linha[0])

        t = image.load_img(img_path, target_size=(nl, nc))
        t = image.img_to_array(t)
        t = np.expand_dims(t, axis=0)
        t = preprocess_input(t)[0]

        x[i] = t
        y[i] = np.float32(linha[1])

    return x, y


def build_model(input_shape):
    """
    Monta o modelo com ResNet50 pré-treinada no ImageNet
    e uma cabeça melhorada para classificação binária.
    """
    base_model = ResNet50(
        weights='imagenet',
        include_top=False,
        input_shape=input_shape
    )

    t = base_model.output
    t = GlobalAveragePooling2D()(t)
    t = BatchNormalization()(t)
    t = Dense(256, activation='relu')(t)
    t = Dropout(0.5)(t)
    predictions = Dense(1, activation="sigmoid")(t)

    model = Model(inputs=base_model.input, outputs=predictions)
    return model, base_model


# ============================================================
# MAIN
# ============================================================
nl = 224
nc = 224
diretorioBd = "."
nome_csv = "todos.csv"

# Carrega as 400 imagens
x, y = leCsv(diretorioBd, nome_csv, nl=nl, nc=nc)

input_shape = (nl, nc, 3)
batch_size = 10

# Número de épocas para cada etapa
epochs_head = 25
epochs_ft = 25

acc_per_repete = []

for repete_no in range(1):
    print("<<<<<<<<<<<<<<<<<<<<<<<<<< Repete indice:", repete_no, "<<<<<<<<<<<<<<<<<<<<<<<<")

    # StratifiedKFold preserva melhor a proporção de classes em cada fold
    kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    acc_per_fold = []
    fold_no = 0

    for a, q in kfold.split(x, y):
        print(f"\n========== Fold {fold_no} ==========")

        x_train_full = x[a]
        y_train_full = y[a]

        x_test = x[q]
        y_test = y[q]

        # separa parte do treino para validação
        x_train, x_val, y_train, y_val = train_test_split(
            x_train_full,
            y_train_full,
            test_size=0.15,
            random_state=42,
            stratify=y_train_full
        )

        # monta modelo
        model, base_model = build_model(input_shape)

        # data augmentation leve
        datagen = ImageDataGenerator(
            width_shift_range=0.05,
            height_shift_range=0.05,
            horizontal_flip=True
        )

        # ====================================================
        # ETAPA 1: TRANSFER LEARNING
        # treina só a cabeça, congelando a ResNet50
        # ====================================================
        for layer in base_model.layers:
            layer.trainable = False

        model.compile(
            optimizer=Adam(learning_rate=1e-4),
            loss='binary_crossentropy',
            metrics=['accuracy']
        )

        ckpt_path = f"best_fold_{repete_no}_{fold_no}.keras"

        callbacks_head = [
            ModelCheckpoint(
                ckpt_path,
                monitor='val_accuracy',
                save_best_only=True,
                verbose=0
            ),
            EarlyStopping(
                monitor='val_accuracy',
                patience=8,
                restore_best_weights=True,
                verbose=0
            ),
            ReduceLROnPlateau(
                monitor='val_loss',
                factor=0.3,
                patience=4,
                min_lr=1e-7,
                verbose=0
            )
        ]

        model.fit(
            datagen.flow(x_train, y_train, batch_size=batch_size),
            epochs=epochs_head,
            verbose=0,
            validation_data=(x_val, y_val),
            callbacks=callbacks_head
        )

        scores_tl = model.evaluate(x_test, y_test, verbose=0)
        acc_tl = scores_tl[1] * 100.0
        print("Acuracidade transflearn de fold %d: %0.4f%%" % (fold_no, acc_tl))

        # ====================================================
        # ETAPA 2: FINE TUNING
        # libera só as últimas 10 camadas da base
        # e recompila com learning rate menor
        # ====================================================
        for layer in base_model.layers[:-10]:
            layer.trainable = False
        for layer in base_model.layers[-10:]:
            layer.trainable = True

        model.compile(
            optimizer=Adam(learning_rate=1e-6),
            loss='binary_crossentropy',
            metrics=['accuracy']
        )

        callbacks_ft = [
            ModelCheckpoint(
                ckpt_path,
                monitor='val_accuracy',
                save_best_only=True,
                verbose=0
            ),
            EarlyStopping(
                monitor='val_accuracy',
                patience=10,
                restore_best_weights=True,
                verbose=0
            ),
            ReduceLROnPlateau(
                monitor='val_loss',
                factor=0.3,
                patience=4,
                min_lr=1e-7,
                verbose=0
            )
        ]

        model.fit(
            datagen.flow(x_train, y_train, batch_size=batch_size),
            epochs=epochs_ft,
            verbose=0,
            validation_data=(x_val, y_val),
            callbacks=callbacks_ft
        )

        # carrega o melhor modelo do fold
        model = keras.models.load_model(ckpt_path)

        scores_ft = model.evaluate(x_test, y_test, verbose=0)
        acc_ft = scores_ft[1] * 100.0
        print("Acuracidade fine tuning de fold %d: %0.4f%%" % (fold_no, acc_ft))

        # guarda a melhor entre TL e FT
        acc_melhor = max(acc_tl, acc_ft)
        acc_per_fold.append(acc_melhor)

        # salva modelo final do fold
        model.save("fei_resnet_5xvalidation_r%d_f%d.keras" % (repete_no, fold_no))

        fold_no += 1

    print("acc_per_fold =", acc_per_fold)
    print("mean_acc=%0.4f%%" % np.mean(acc_per_fold))

    acc_per_repete.append(np.mean(acc_per_fold))

print("<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<")
print("acc_per_repete =", acc_per_repete)
print("mean_acc=%0.4f%%" % np.mean(acc_per_repete))

<<<<<<<<<<<<<<<<<<<<<<<<<< Repete indice: 0 <<<<<<<<<<<<<<<<<<<<<<<<

========== Fold 0 ==========
Acuracidade transflearn de fold 0: 95.0000%
Acuracidade fine tuning de fold 0: 95.0000%

========== Fold 1 ==========
Acuracidade transflearn de fold 1: 100.0000%
Acuracidade fine tuning de fold 1: 100.0000%

========== Fold 2 ==========
Acuracidade transflearn de fold 2: 95.0000%
Acuracidade fine tuning de fold 2: 97.5000%

========== Fold 3 ==========
Acuracidade transflearn de fold 3: 100.0000%
Acuracidade fine tuning de fold 3: 100.0000%

========== Fold 4 ==========
Acuracidade transflearn de fold 4: 93.7500%
Acuracidade fine tuning de fold 4: 96.2500%
acc_per_fold = [94.9999988079071, 100.0, 97.50000238418579, 100.0, 96.24999761581421]
mean_acc=97.7500%
<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
acc_per_repete = [np.float64(97.74999976158142)]
mean_acc=97.7500%
